In [ ]:
!nvidia-smi

In [ ]:
# Paths come from path_config.yaml; edit that file's current_workstation to switch machines.
from path_config import PMOF_CODE_DIR, DATA_BASE_DIR, DATA_ACTIONCLS_DIR

from ultralytics import YOLO
import albumentations as A
import torch
from pathlib import Path
import cv2

In [ ]:
pretrained_pmof_model = PMOF_CODE_DIR / 'benchmarks/runs/obb/20-m-1024-CCa18PPa18-P/weights/epoch7.pt'
model = YOLO("yolo26m-obb.yaml").load(pretrained_pmof_model)
model.info()

In [ ]:
custom_transforms = [
#         ### A1: Geometric Invariance
         A.Resize(width=320, height=320),
#         A.Affine(scale=[1,1.2], rotate=[0,90], shear=0, translate_percent=0, keep_ratio=True, p=0.5),
         A.Rotate(limit=90, p=1, border_mode=cv2.BORDER_CONSTANT), #border to be black       
         A.HorizontalFlip(p=0.5),
         A.VerticalFlip(p=0.5),
#         ### A2: Occlusion
         A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(0.1, 0.25),
                     hole_width_range=(0.1, 0.25), p=0.5), #fill_value=0
        ### A3 & A7: ChannelDropout/Greyscale
        A.OneOf([
            A.ToGray(p=1.0), # p=1.0 inside OneOf
            A.ChannelDropout(p=1.0) # p=1.0 inside OneOf
        ], p=0.2), # Apply one of these 20% of the time
        A.OneOf([A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.8),            
                 A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.8),            
                 A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.8),            
                 A.RandomGamma(gamma_limit=(80, 120), p=0.8),        
                ], p=0.7),
]


results = model.train(data=str(DATA_BASE_DIR / "PMOF_action.yaml"), epochs=1, patience=5, name='obb', imgsz=320,
                      optimizer='SGD', lr0=0.001, momentum=0.9, weight_decay=0.0005,
                      hsv_h=0, hsv_s=0, hsv_v=0, degrees=0, flipud=0, fliplr=0, 
                      translate=0, scale=0, mosaic=0, mixup=0, erasing=0, close_mosaic=1,
                      val=True, iou=0.50,
                    augmentations=[custom_transforms]) #apply no additional augmentations